# Would I agree with the flag if I looked at the trains myself?

In the [measurement notebook](03_mta_disruptions_measured_v3_1.ipynb) I used train counts to flag places where subway service dropped. Now I want to look at a few of these cases myself, as if the flag hadn't decided anything yet. I read the alert, check routes and stations, and look at the trains recorded around that time.

One limit: I read the same train records that fed the rule. So I can catch mistakes in how I used them, but if the feed missed a train, I'll miss it too.


## Which version of the measurement am I checking?

I use the weekday daytime sample from the `period_3_5` run, the same version I use in the [pickup analysis](08_subway_fhv_causal_analysis_storytelling.ipynb). Every verdict should point back to the run that made the flag.


In [ ]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*'generic' unit for NumPy timedelta.*")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 220)


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Open this notebook inside the project containing pyproject.toml.")


PROJECT_ROOT = find_project_root()

# Which measurement run to validate (the primary variant)
VARIANT = "period_3_5"
RUN_MODE = "full"

# Sample design: the main analysis is weekday daytime, so that is what I validate
PERIODS = ("weekday_daytime",)   # add "night_or_weekend" only if night results enter the analysis
SAMPLE_PER_STRATUM = 10    # per treatment tier
NOT_FLAGGED_SAMPLE = 10    # zones served by a disrupted route with no flag at all
SAMPLE_SEED = 2026
CAUSE_GROUPS = ("main",)   # the main analysis sample

# Display window, matching notebook 03
LOCAL_TIMEZONE = "America/New_York"
PRE_WINDOW = pd.Timedelta(150, unit="min")
POST_WINDOW = pd.Timedelta(60, unit="min")
REFERENCE_WEEKS = (1, 2)   # comparison days shown next to the episode day

SILVER_DIR = PROJECT_ROOT / "data" / "silver" / "mta_disruptions"
TRAIN_REDUCED_DIR = PROJECT_ROOT / "data" / "interim" / "subwaydata_nyc" / "trip_stops_v2"
STATIONS_PATH = PROJECT_ROOT / "data" / "external" / "mta_subway_stations" / "mta_subway_stations.csv"
VALIDATION_DIR = PROJECT_ROOT / "data" / "validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
LABELS_PATH = VALIDATION_DIR / f"validation_labels_{VARIANT}_{'_'.join(PERIODS)}.csv"

## Finding the run behind the flags

The latest-run pointer can change when I rebuild the measurement. So I print the run name and settings fingerprint before reading the tables, to know exactly which version I'm looking at.


In [ ]:
run_name = (SILVER_DIR / f"latest_{RUN_MODE}_{VARIANT}.txt").read_text(encoding="utf-8").strip()
RUN_DIR = SILVER_DIR / run_name
manifest = json.loads((RUN_DIR / "run_manifest.json").read_text(encoding="utf-8"))
print("Validating run:", run_name, "| settings fingerprint:", manifest.get("settings_fingerprint"))


def as_tuple(value):
    return tuple(value) if isinstance(value, (list, np.ndarray)) else value


event_study = pd.read_parquet(RUN_DIR / "event_study.parquet")
attributed_bins = pd.read_parquet(RUN_DIR / "attributed_bins.parquet")
episodes = pd.read_parquet(RUN_DIR / "disruption_episodes.parquet")
bridge = pd.read_parquet(RUN_DIR / "station_complex_zone_bridge.parquet")
for column in ["routes", "disrupted_routes", "observed_effects", "onset_effects"]:
    episodes[column] = episodes[column].map(as_tuple)

# Station list: realtime parent stop -> complex, and readable complex names
stations = pd.read_csv(STATIONS_PATH, dtype=str)
parent_to_complex = dict(zip(stations["GTFS Stop ID"].str.strip(), stations["Complex ID"].str.strip()))
complex_names = stations.groupby(stations["Complex ID"].str.strip())["Stop Name"].agg(lambda n: " / ".join(sorted(set(n))))
daytime_routes = stations.assign(route=stations["Daytime Routes"].fillna("").str.split()).explode("route")
zone_names = bridge.drop_duplicates("LocationID").set_index("LocationID")["zone_name"]

print("Event-study rows:", len(event_study), "| flagged bins:", len(attributed_bins), "| episodes:", len(episodes))

Validating run: full_period_3_5_20261012_034129 | settings fingerprint: e026eb145098


Event-study rows: 83119 | flagged bins: 614766 | episodes: 9373


## Looking for missed losses as well as wrong flags

If I only review flagged places, I can find false alarms but never the losses the rule missed. So I sample from several levels of loss and also from unflagged zones that the affected routes usually serve. I use a fixed seed for the first draw and keep the same sample once I start recording verdicts.


In [ ]:
def broad_period(period):
    return "weekday_daytime" if period in ("am_peak", "midday", "pm_peak") else "night_or_weekend"


STATION_LIST_SYMBOL = {"6X": "6", "7X": "7", "FX": "F", "GS": "S", "FS": "S", "H": "S", "SI": "SIR", "SS": "SIR"}
route_zones = {
    route: set(bridge.loc[bridge["station_complex_id"].isin(
        daytime_routes.loc[daytime_routes["route"].eq(route), "Complex ID"].str.strip()), "LocationID"])
    for route in daytime_routes["route"].dropna().unique()
}


def zones_served(routes):
    return set().union(*(route_zones.get(STATION_LIST_SYMBOL.get(r, r), set()) for r in routes)) if routes else set()


if LABELS_PATH.exists():
    sample = pd.read_csv(LABELS_PATH, dtype={"verdict": "string", "notes": "string"}, keep_default_na=False)
    print(f"Loaded existing sample with {len(sample)} rows and {sample['verdict'].ne('').sum()} verdicts")
else:
    rng = np.random.default_rng(SAMPLE_SEED)
    flagged = event_study.loc[event_study["cause_eligibility"].isin(CAUSE_GROUPS)].copy()
    flagged["broad_period"] = flagged["period"].map(broad_period)
    flagged = flagged.loc[flagged["broad_period"].isin(PERIODS)]
    flagged["stratum"] = flagged["treatment_tier"] + " | " + flagged["broad_period"]
    parts = []
    for stratum, rows in flagged.groupby("stratum"):
        take = min(SAMPLE_PER_STRATUM, len(rows))
        parts.append(rows.iloc[rng.choice(len(rows), size=take, replace=False)])
    flagged_sample = pd.concat(parts, ignore_index=True)[["episode_key", "LocationID", "stratum", "period"]]

    # Unflagged: a measured main-cause episode and a zone its disrupted routes serve, with no flag there
    measured = episodes.loc[episodes["measure_status"].eq("measured") & episodes["within_study_window"]
                            & episodes["cause_eligibility"].isin(CAUSE_GROUPS)
                            & episodes["period"].map(broad_period).isin(PERIODS)]
    flagged_pairs = set(zip(event_study["episode_key"], event_study["LocationID"]))
    unflagged_rows = []
    for index in rng.permutation(len(measured)):
        ep = measured.iloc[index]
        candidates = sorted(z for z in zones_served(ep["disrupted_routes"]) if (ep["episode_key"], z) not in flagged_pairs)
        if candidates:
            unflagged_rows.append({"episode_key": ep["episode_key"], "LocationID": int(rng.choice(candidates)),
                                   "stratum": "not_flagged | " + broad_period(ep["period"]), "period": ep["period"]})
        if len(unflagged_rows) == NOT_FLAGGED_SAMPLE:
            break

    sample = pd.concat([flagged_sample, pd.DataFrame(unflagged_rows)], ignore_index=True)
    sample.insert(0, "sample_id", np.arange(1, len(sample) + 1))
    sample["zone_name"] = sample["LocationID"].map(zone_names)
    sample["verdict"] = ""
    sample["notes"] = ""
    sample.to_csv(LABELS_PATH, index=False)
    print(f"Drew a new sample of {len(sample)} rows -> {LABELS_PATH}")

display(sample.groupby("stratum").size().rename("rows").to_frame())

Loaded existing sample with 50 rows and 2 verdicts


,rows
stratum,
access_lost | weekday_daytime,10
diverted_only | weekday_daytime,10
not_flagged | weekday_daytime,10
partial_lost | weekday_daytime,10
route_lost | weekday_daytime,10


## How I decide whether a flag is supported

Say a reroute alert covers several stations. If the affected line disappears from one station during the alert but another line still runs there, I expect a route-loss flag. If the table says riders lost all service in that direction, I want evidence for that stronger claim.

To see what normal looks like, I compare with the same time one and two weeks earlier, keeping in mind these days can have their own problems. I record `supported`, `not_supported` or `unclear` and write down why. For an unflagged zone, `supported` means the evidence agrees it should stay unflagged.

The prepared snapshot has the saved labels and measurement tables. To look at trains for a new example, I need the daily reduced train files in `data/interim/subwaydata_nyc/trip_stops_v2`.


In [ ]:
BIN = pd.Timedelta(15, unit="min")


def local_wall(ts):
    return pd.Timestamp(ts).tz_convert(LOCAL_TIMEZONE).tz_localize(None)


def recorded_trains(start_wall, end_wall, routes, complexes, align=pd.Timedelta(0)):
    """Trains per route, direction, complex, and local 15-minute bin, read from the raw trip-stop files.
    `align` moves comparison days onto the episode day's timeline, so columns line up in real time order."""
    days = pd.date_range(start_wall.normalize() - pd.Timedelta(days=1), end_wall.normalize() + pd.Timedelta(days=1), freq="D")
    frames = [pd.read_parquet(TRAIN_REDUCED_DIR / f"{d:%Y-%m-%d}.parquet") for d in days
              if (TRAIN_REDUCED_DIR / f"{d:%Y-%m-%d}.parquet").exists()]
    if not frames:
        return pd.DataFrame(columns=["route_id", "direction", "station_complex_id", "bin", "trains"])
    stops = pd.concat(frames, ignore_index=True)
    stops = stops.loc[stops["route_id"].isin(routes)].copy()
    stops["station_complex_id"] = stops["parent"].map(parent_to_complex)
    stops = stops.loc[stops["station_complex_id"].isin(complexes)]
    stops["wall"] = pd.to_datetime(stops["time_s"], unit="s", utc=True).dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)
    stops = stops.loc[stops["wall"].ge(start_wall) & stops["wall"].lt(end_wall)]
    stops["bin"] = stops["wall"].dt.floor("15min") + align
    return (stops.groupby(["route_id", "direction", "station_complex_id", "bin"])["trip_uid"].nunique()
            .rename("trains").reset_index())


def time_columns(table):
    """Columns in real time order, labeled by clock time."""
    table = table.reindex(columns=sorted(table.columns))
    return table.rename(columns=lambda t: t.strftime("%H:%M"))


def station_list_symbol(route):
    return STATION_LIST_SYMBOL.get(route, route)


def show_sample(sample_id):
    row = sample.loc[sample["sample_id"].eq(sample_id)].iloc[0]
    ep = episodes.loc[episodes["episode_key"].eq(row["episode_key"])].iloc[0]
    zone_complexes = set(bridge.loc[bridge["LocationID"].eq(row["LocationID"]), "station_complex_id"])
    routes = tuple(ep["routes"])

    print(f"Sample {sample_id} | stratum: {row['stratum']} | zone {row['LocationID']} {row['zone_name']}")
    print(f"Episode {ep['episode_key']} | routes {', '.join(routes)} (disrupted: {', '.join(ep['disrupted_routes'])})"
          f" | cause {ep['cause']} | effects {', '.join(ep['observed_effects'])}")
    print(f"First alert {local_wall(ep['incident_start']):%a %Y-%m-%d %H:%M} | onset {local_wall(ep['onset_time']):%H:%M}"
          f" | announced end {local_wall(ep['alert_end']):%Y-%m-%d %H:%M}"
          f"{' (assumed, no end update)' if ep['is_right_censored'] else ''}")
    text = str(ep["episode_text"])
    print("Alert text:", text[:700])
    route_stations = daytime_routes.loc[daytime_routes["route"].isin({station_list_symbol(r) for r in routes}), "Stop Name"].unique()
    named = sorted(n for n in route_stations if len(n) >= 4 and re.search(rf"(?<!\w){re.escape(n)}(?!\w)", text, re.I))
    print("Stations of these routes named in the alert:", ", ".join(named) or "none found")

    # Where the pipeline flagged this episode, along the whole route (rows at this zone marked with *)
    episode_flags = attributed_bins.loc[attributed_bins["episode_key"].eq(row["episode_key"])].copy()
    if len(episode_flags):
        episode_flags["type"] = np.select([episode_flags["access_lost"], episode_flags["lost"],
                                           episode_flags["partial_lost"], episode_flags["diverted"]],
                                          ["access", "route", "partial", "diversion"], "")
        overview = (episode_flags.groupby(["route_id", "direction", "station_complex_id"])
                    .agg(first=("bin_utc", "min"), last=("bin_utc", "max"), bins=("bin_utc", "size"),
                         types=("type", lambda t: ", ".join(sorted(set(t)))))
                    .reset_index())
        overview["station"] = overview["station_complex_id"].map(complex_names).fillna(overview["station_complex_id"])
        overview["this_zone"] = np.where(overview["station_complex_id"].isin(zone_complexes), "*", "")
        overview["first"] = overview["first"].dt.tz_convert(LOCAL_TIMEZONE).dt.strftime("%H:%M")
        overview["last"] = (overview["last"] + BIN).dt.tz_convert(LOCAL_TIMEZONE).dt.strftime("%H:%M")
        print(f"\nAll flagged stations of this episode ({len(overview)} route x direction x station series):")
        display(overview[["this_zone", "route_id", "direction", "station", "first", "last", "bins", "types"]]
                .sort_values(["route_id", "direction", "first"]).head(40))

    # Flags set by the pipeline at this zone's stations
    flags = attributed_bins.loc[attributed_bins["episode_key"].eq(row["episode_key"])
                                & attributed_bins["station_complex_id"].isin(zone_complexes)].copy()
    if flags.empty:
        print("\nPipeline flags at this zone: none")
    else:
        flags["bin"] = flags["bin_utc"].dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)
        flags["mark"] = np.select([flags["access_lost"], flags["lost"], flags["partial_lost"], flags["diverted"]],
                                  ["NO_ACCESS", "ROUTE_LOST", "PARTIAL", "DIV"], "")
        flags["station"] = (flags["route_id"] + " " + flags["direction"] + " | "
                            + flags["station_complex_id"].map(complex_names).fillna(flags["station_complex_id"]))
        print("\nPipeline flags at this zone:")
        display(time_columns(flags.pivot_table(index="station", columns="bin", values="mark", aggfunc="first").fillna("")))

    # Independent view: recorded trains on the episode day and the same clock times earlier
    start_wall = local_wall(ep["incident_start"] - PRE_WINDOW).floor("15min")
    end_wall = local_wall(ep["alert_end"] + POST_WINDOW).ceil("15min")
    key = ["route_id", "direction", "station_complex_id", "bin"]
    if not any(TRAIN_REDUCED_DIR.glob("*.parquet")):
        print("Train-stop inspection needs the daily reduced train files in data/interim/subwaydata_nyc/trip_stops_v2.")
        return

    days = {"episode": recorded_trains(start_wall, end_wall, routes, zone_complexes)}
    for weeks in REFERENCE_WEEKS:
        shift = pd.Timedelta(days=7 * weeks)
        days[f"-{weeks}w"] = recorded_trains(start_wall - shift, end_wall - shift, routes, zone_complexes, align=shift)
    combined = None
    for label, frame in days.items():
        frame = frame.set_index(key)["trains"].rename(label)
        combined = frame.to_frame() if combined is None else combined.join(frame, how="outer")
    if combined is None or combined.empty:
        print("\nNo recorded trains of these routes at this zone's stations on any of the days.")
        return
    combined = combined.fillna(0).astype(int).reset_index()
    combined["cell"] = combined[list(days)].astype(str).agg(" / ".join, axis=1)
    combined["station"] = (combined["route_id"] + " " + combined["direction"] + " | "
                           + combined["station_complex_id"].map(complex_names).fillna(combined["station_complex_id"]))
    print(f"\nRecorded trains per 15 min ({' / '.join(days)}), from the raw trip-stop files:")
    display(time_columns(combined.pivot_table(index="station", columns="bin", values="cell", aggfunc="first")
                         .fillna(" / ".join(["0"] * len(days)))))


def record_verdict(sample_id, verdict, notes=""):
    """Saving one verdict to the label file."""
    assert verdict in ("supported", "not_supported", "unclear"), "verdict must be supported, not_supported, or unclear"
    sample.loc[sample["sample_id"].eq(sample_id), ["verdict", "notes"]] = [verdict, notes]
    sample.to_csv(LABELS_PATH, index=False)
    remaining = sample["verdict"].eq("").sum()
    print(f"Saved sample {sample_id}: {verdict}. {remaining} rows left to review.")

## Reviewing one case at a time

I leave `SAMPLE_ID` empty to get a case I haven't labeled yet. After reading the evidence I record the verdict in the next cell. When I only want to rerun the summaries, I don't touch that cell.


In [ ]:
SAMPLE_ID = None
if SAMPLE_ID is None:
    unlabeled = sample.loc[sample["verdict"].eq(""), "sample_id"]
    SAMPLE_ID = int(unlabeled.iloc[0]) if len(unlabeled) else int(sample["sample_id"].iloc[0])
show_sample(SAMPLE_ID)

Sample 3 | stratum: access_lost | weekday_daytime | zone 51 Co-Op City
Episode 169314-1 | routes 2, 5 (disrupted: 2, 5) | cause medical_emergency | effects stop_skipping


First alert Wed 2024-09-11 11:13 | onset 11:24 | announced end 2024-09-11 11:40
Alert text: Northbound 2 5 trains are running on the express track from 3 Av-149 St to E 180 St. We're requesting EMS assistance for someone in need on a train at E 180 St.
Stations of these routes named in the alert: 3 Av-149 St, E 180 St

All flagged stations of this episode (126 route x direction x station series):


,this_zone,route_id,direction,station,first,last,bins,types
0,,2,N,72 St,10:15,11:00,3,access
1,,2,N,34 St-Penn Station,10:15,11:00,3,access
2,,2,N,Chambers St,10:15,11:00,3,access
3,,2,N,Wall St,10:15,11:00,3,access
4,,2,N,Clark St,10:15,11:00,3,access
5,,2,N,Hoyt St,10:15,11:00,3,access
6,,2,N,President St-Medgar Evers College,10:15,11:00,3,"access, route"
7,,2,N,Beverly Rd,10:15,11:00,3,access
8,,2,N,Newkirk Av-Little Haiti,10:15,11:00,3,access
9,,2,N,Flatbush Av-Brooklyn College,10:15,11:00,3,access



Pipeline flags at this zone:


bin,10:15,10:30,10:45,11:00
station,,,,
5 S | Baychester Av,NO_ACCESS,NO_ACCESS,NO_ACCESS,PARTIAL


Train-stop inspection needs the daily reduced train files in data/interim/subwaydata_nyc/trip_stops_v2.


In [ ]:
# After looking at the row above, record the verdict with your own note. Format:
record_verdict(2, "supported", "J N 0 trains at Kosciuszko/Gates/Halsey 16:45-17:30 vs 1-3, inside the alert's Myrtle Av-Crescent St segment; starts before first alert; Gates route_lost after 17:45 overlaps J/Z skip-stop hours (weak); M Central Av partial marginal")

Saved sample 2: supported. 48 rows left to review.


## What I can take from the review so far

So far I reviewed **2 of 50 rows**, and both supported the weekday daytime access-loss flag. Two agreements is a start, but it's too few to say how often the rule is right or how many losses it misses.

I also compare how long alerts lasted with how long the measured loss lasted. In the weekday daytime episodes below, the median announced duration is **18 minutes**, but the median measured span is **105 minutes** where a span exists. These medians come from different numbers of episodes, so I don't read the difference as a typical gap for one incident. Still, it shows why I keep both clocks when I choose the timing anchor in the [pickup analysis](08_subway_fhv_causal_analysis_storytelling.ipynb).


In [ ]:
def wilson_interval(successes, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = successes / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (centre - half, centre + half)


labelled = sample.loc[sample["verdict"].ne("")]
summary_rows = []
for stratum, rows in labelled.groupby("stratum"):
    decided = rows.loc[rows["verdict"].ne("unclear")]
    supported = int(decided["verdict"].eq("supported").sum())
    low, high = wilson_interval(supported, len(decided))
    summary_rows.append({"stratum": stratum, "reviewed": len(rows), "unclear": int(rows["verdict"].eq("unclear").sum()),
                         "supported": supported, "not_supported": len(decided) - supported,
                         "supported_share": supported / len(decided) if len(decided) else np.nan,
                         "ci_low": low, "ci_high": high})

print(f"Reviewed {len(labelled)} of {len(sample)} sampled rows")
if summary_rows:
    display(pd.DataFrame(summary_rows).round(3))

Reviewed 2 of 50 sampled rows


,stratum,reviewed,unclear,supported,not_supported,supported_share,ci_low,ci_high
0,access_lost | weekday_daytime,2,0,2,0,1.0,0.342,1.0


In [ ]:
weekday_daytime = ["am_peak", "midday", "pm_peak"]
ep = episodes.loc[episodes["within_study_window"] & episodes["measure_status"].eq("measured")
                  & episodes["cause_eligibility"].eq("main") & episodes["period"].isin(weekday_daytime)].copy()
ep["alert_minutes"] = (ep["alert_end"] - ep["onset_time"]).dt.total_seconds() / 60

measured_span = event_study.groupby("episode_key").agg(start=("measured_start_utc", "min"), end=("measured_end_utc", "max"))
ep = ep.merge(measured_span, left_on="episode_key", right_index=True, how="left")
ep["measured_minutes"] = (ep["end"] - ep["start"]).dt.total_seconds() / 60

print("Main-sample episodes (weekday daytime, measured, main causes):", len(ep))
display(ep[["alert_minutes", "measured_minutes"]].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(1))

display(pd.DataFrame({
    f">= {t} min": {
        "by announced duration": int((ep["alert_minutes"] >= t).sum()),
        "by announced duration, not censored": int(((ep["alert_minutes"] >= t) & ~ep["is_right_censored"]).sum()),
        "by measured duration": int((ep["measured_minutes"] >= t).sum()),
    }
    for t in (30, 60, 120)
}))

Main-sample episodes (weekday daytime, measured, main causes): 2997


,alert_minutes,measured_minutes
count,2997.0,2312.0
mean,40.3,121.4
std,62.4,77.3
min,1.0,15.0
25%,9.0,60.0
50%,18.0,105.0
75%,41.0,150.0
90%,108.0,210.0
max,1323.0,570.0


,>= 30 min,>= 60 min,>= 120 min
by announced duration,1011,540,268
"by announced duration, not censored",886,415,143
by measured duration,2277,1867,1084
